<style>

.footnote-list {
    display: none;
}
</style>

# Train a Basic BPE Tokenizer
A trained BPE takes an input text, splits it up into tokens, and assigns a token ID (not to be confused with the numerical vector embeddings of said token):

```{figure} ../../figures/class1/004_BPE.png
---
name: BPE-high-level
---
Modified from {cite:t}`build-llms-from-scratch-book`s GitHub repo under the [Apache License](https://github.com/rasbt/LLMs-from-scratch/blob/main/LICENSE.txt). 
```

We won't be looking at the finished BPE, but on a simple training workflow!

## 4.1 Training
We begin the process with an initial vocabulary, comprising a set of all unique individual characters or bytes in our corpus.

Then we do these these steps:
1. **Count** the pairs of existing tokens in our corpus (**characters** *or* **bytes**[^bytes_ex]) to **find the most frequent pair**
2. **Merge** the most frequent pair (to become a new token) & **replace** it with new token ID 
3. **Repeat** 1 & 2 until there are no gains (or until we have hit "max" vocabulary length)


:::{admonition} "Pairs" Clarification
:class: important
I use "pair" to mean **two adjacent tokens**. At the start, this is may be individual characters or bytes (e.g., the pair "t" + "h"), but new tokens quickly become combinations beyond single characters or bytes (e.g., "th" + "e" which can be merged to form "the").
:::

[^bytes_ex]: LLMs use Byte-level BPE, not character-level. We'll start with words for intution, then transition to bytes.

### Step 1: Finding Frequent Combinations
Let's start with an example of a simple sequence:


In [1]:
input_text = "the cat sat on the mat"

In Python, we can easily iterate over characters in a string. Let's do that with `enumerate` to also get the position of the character:

In [2]:
for i, char in enumerate(input_text):
    print(i, char)

0 t
1 h
2 e
3  
4 c
5 a
6 t
7  
8 s
9 a
10 t
11  
12 o
13 n
14  
15 t
16 h
17 e
18  
19 m
20 a
21 t


You might already be able to spot adjacent tokens that seem more frequent than others! Give it a go. 
:::{admonition} QUESTION
:class: red
Can you spot the most frequent pair just by looking at it? 

</details>
<summary>Click to see ANSWER</summary>
It's "a + t" ("cat", "sat", "mat"), followed by the strong contender "t + h" (2x "the")
</details>
:::

#### Construct Pairs
A real corpus won't only have one sentence (that would also not make a good BPE tokenizer). We therefore need to construct pairs automatically. 

To do this, we need to go through each `char` and "connect" it to the `next_char`:
```
iteration 1: 
    char = t 
    next_char = h
    pair = (char, next_char)
iteration 2: 
    char = h 
    next_char = e
    pair = (char, next_char)
```

:::{admonition} HANDS-ON: Identify the `next_char` & create pairs!
:class: red
Loop over characters with `enumerate`: 
1. Identify `char` and `next_char`
2. Save these as a pair 
3. Print the pair
4. Remember to end the for loop *before* you check the last character, since that one won't have a consecutive pair to check with!
:::: 

##### Solution

:::{admonition} How to identify `next_char`
:class: tip, dropdown
Each character in a string has a position. We can select each character with this position e.g., text[2] would be "e"[^zero_index]. With `enumerate`, we've made that position explicit using `i, char`. `i` is the current char, how would we get the next one?

<details>
<summary>Click to see answer</summary>
We identify the <code>next_char</code> by adding +1 to <code>i</code>, i.e., <code>text[i + 1]</code>
</details>
::::

[^zero_index]: Remember that Python is zero-indexed!

:::{admonition} End the loop before the last character.
:class: tip, dropdown
We can select which parts to loop over using **slicing** on any sequence (e.g., string, list, tuple).

We use the syntax `input_text[start:stop]`, where we leave `start` empty (take from beginning) but indicate that it should `stop` at `-1`to exclude the last token:
```python
for i, char in enumerate(input_text[:-1]): # select everything but the last thing (-1)
    # do something
```

For more info on slicing: 
- See [TowardsDataScience on "basics"](https://towardsdatascience.com/slicing-in-python-a-comprehensive-guide-a609c3bb877c/)
- See [The Python Coding Stack on EVERYTHING you can do with slicing](https://www.thepythoncodingstack.com/p/a-python-slicing-story)
::::

In [3]:
for i, char in enumerate(input_text[:-1]):
    next_char = input_text[i + 1]
    pair = (char, next_char)
    print(pair)

('t', 'h')
('h', 'e')
('e', ' ')
(' ', 'c')
('c', 'a')
('a', 't')
('t', ' ')
(' ', 's')
('s', 'a')
('a', 't')
('t', ' ')
(' ', 'o')
('o', 'n')
('n', ' ')
(' ', 't')
('t', 'h')
('h', 'e')
('e', ' ')
(' ', 'm')
('m', 'a')
('a', 't')


#### Count Pairs
We shouldn't just print the pairs, but record each unique pair and the frequency with which they occur to a `dictionary`. Below is a starting point:

In [4]:
def count_pairs(input_data):
    """"
    A function which takes input_data (string or byte-encoded), constructs a dictionary of unique consecutive pairs 
    & counts their occurence in input data.
    """
    # initialize empty counts dictoinary
    counts = {} 

    # your code

    return counts

:::{admonition} HANDS-ON
:class: red
Finish the function above. 

Your function should loop over `input_text`, constructing pairs and counting them in a `counts` dictionary. 

This dictionary has each **unique pair** as a `key` and their frequency as a `value`. In other words, this is the end goal:
```
counts = {('t', 'h'): 2, ('h', 'e'): 2, ('e', ' '): 2, (' ', 'c'): 1, ('c', 'a'): 1, ('a', 't'): 3, ...}
```
:::

##### Solution

:::{admonition} Incrementing a value in a dictionary
:class: tip, dropdown
See this [geeksforgeeks](https://www.geeksforgeeks.org/python/python-increment-value-in-dictionary/) guide !
::::

In [5]:
def count_pairs(input_data: str | list): # takes string "He is happy" or [72, 101, 32, 105, 115, 32, 104, 97, 112, 112, 121]
    """"
    A function which takes input_data (string or list of bytes), constructs a dictionary of unique consecutive pairs 
    & counts their occurence in input data.
    """
    # initialize empty counts dictoinary
    counts = {} 

    # loop over each char in input text except the last one!
    for i, char in enumerate(input_data[:-1]):
        next_char = input_data[i + 1]
        pair = (char, next_char)

        try: # try to increment a value to a key that already exists
            counts[pair] += 1 
        except KeyError: # if key does not exist, it'll throw a key error (you won't be able to increment it). But you can create it!
            counts[pair] = 1

    return counts

:::{admonition} Solution for *corpus* rather than a single sentence.
:class: tip, dropdown
The rest of the tutorial works only on a sentence, but in reality we would work on a `corpus`
```python
corpus = [
    "My cat is sad", 
    "The cat sat on the mat", 
    "The dog hates the cat",
]
```
This would require an outer-loop:
```python
def count_pairs(input_data: list[str | list]): # takes list of strings or bytes!
    """"
    A function which takes input_data (list of string or list of bytes), constructs a dictionary of unique consecutive pairs 
    & counts their occurence in input data.
    """
    # initialize empty counts dictoinary
    counts = {} 

    # outer loop!
    for sequence in input_data:
        # loop over each char in input text except the last one!
        for i, char in enumerate(sequence[:-1]):
            next_char = sequence[i + 1]
            pair = (char, next_char)

            try: # try to increment a value to a key that already exists
                counts[pair] += 1 
            except KeyError: # if key does not exist, it'll throw a key error (you won't be able to increment it). But you can create it!
                counts[pair] = 1

    return counts
```

**Ignore this for now**. We will be working on one sequence for the entire exercise 4. You can tinker with this after doing the basic tutorial!
::::

#### Encode Text!

The example above used *characters*. Computers represent these with the **Unicode** standard, which assigns a unique number (**a code point**) to each character, no matter language or symbol. More than 150,000 codepoints[^uni_explorer] exist. For example, the Danish Å, Ø, Æ are represented as:
```
Å = U+00C5
Ø = U+00D8
Æ = U+00C6
```

Rather than initializing BPE with a vocabulary of +150000 code points, we encode our text into **UTF-8**, an encoding that stores these characters as **single bytes** or **sequences of bytes**:
```python
Å = [195, 133] # 2 bytes!
Ø = [195, 152]
Æ = [195, 134]

A = [65] # 1 byte!
```

Each byte has up to 256 possible values (from 0 to 255)[^optional_ex], so we reduce our initial "base" vocabulary size drastically by using these values as our initial tokens in **Byte-level BPE**. Above token number 255, we begin creating tokens by merging bytes!

:::{admonition} OPTIONAL: Why 256 values? What is a byte actually?
:class: tip, dropdown
A **byte** is a series of **8 bits**, where each bit can hold a binary number 0 or 1 (e.g., 0110 0110). This means a single byte can represent $2^8 = 256$ possible *decimal* values, ranging from 0 to 255. 

Let's break this down. Think of each bit as a light switch holding 2 possible values:
```
bit 1:  0 or 1
bit 2:  0 or 1
bit 3:  0 or 1
bit 4:  0 or 1
bit 5:  0 or 1
bit 6:  0 or 1
bit 7:  0 or 1
bit 8:  0 or 1
```

With 2 possible values for each bit, we get 256 possible combinations for 8 bits:
```
2 × 2 × 2 × 2 × 2 × 2 × 2 × 2 = 2^8 = 256
```

We can also represent these positiotns as powers of 2, starting at 2^0 on the right: 
| Position | 7 | 6 | 5 | 4 | 3 | 2 | 1 | 0 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Power of 2 | \(2^7\) | \(2^6\) | \(2^5\) | \(2^4\) | \(2^3\) | \(2^2\) | \(2^1\) | \(2^0\) |
| Value | 128 | 64 | 32 | 16 | 8 | 4 | 2 | 1 |

For example, the decimal value "1" is represented in binary as:
```
0 0 0 0 0 0 0 1
```
The 1 is in the 2^0 position, and 2^0 is equal to 1.

If you want to geek about about binary systems and how to convert from decimal numbers (e.g,. 120) (eg., 0110 0110). I highly recommend these videos from Khan Academy:
1. [The binary system](https://www.khanacademy.org/computing/computers-and-internet/xcae6f4a7ff015e7d:digital-information/xcae6f4a7ff015e7d:binary-numbers/v/the-binary-number-system?referrer=share_link)
2. [Convert decimial numbers to binary](https://www.khanacademy.org/computing/computers-and-internet/xcae6f4a7ff015e7d:digital-information/xcae6f4a7ff015e7d:binary-numbers/v/converting-decimal-numbers-to-binary?referrer=share_link).
::::

[^uni_explorer]: You can explore these codepoints on [unicode-explorer.com](https://unicode-explorer.com/).
[^optional_ex]: I initally went on a *long* tangent about bits & bytes here, but decided against it. Read more about it in the optional tip box.

**In code**, we convert text by using the `encode("utf.8")` method on our string. We wrap it in a list to get the bytes out:

In [6]:
token_ids = list(input_text.encode("utf-8"))
print(token_ids)

[116, 104, 101, 32, 99, 97, 116, 32, 115, 97, 116, 32, 111, 110, 32, 116, 104, 101, 32, 109, 97, 116]


Our count function will work just fine, even with bits!

In [7]:
counts = count_pairs(token_ids)
print(counts)

{(116, 104): 2, (104, 101): 2, (101, 32): 2, (32, 99): 1, (99, 97): 1, (97, 116): 3, (116, 32): 2, (32, 115): 1, (115, 97): 1, (32, 111): 1, (111, 110): 1, (110, 32): 1, (32, 116): 1, (32, 109): 1, (109, 97): 1}


##### Decode (revert back)
If we want to check how this works, we can use the `chr()` to **decode** the bytes. Here for the first byte-pair:

In [8]:
print(chr(116), (chr(104)))

t h


### Finding the most frequent pair
Below, I've created a function that finds the most frequent pair when given a list of tokens! We can re-use our counts, and add this functionality on top. We can do this with the `max()` function[^geeks_max]:

[^geeks_max]: See [https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/](https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/).

In [9]:
# i've started calling it token_ids, since we're always working with byte-sized stuff!
def find_most_frequent_pair(token_ids):
    counts = count_pairs(token_ids)
    most_frequent_pair = max(counts, key=counts.get)

    return most_frequent_pair

### Step 2. Merge & Replace!
Now that we have a most frequent pair, we're ready to merge this, replacing it with a token id. For our first new token, we'll give it the id `256`! 

In [10]:
def merge_pair(token_ids:list, pair:tuple, new_id:int):
    """
    Replace every copy of pair with new_id
    """
    new_token_ids = []

    # iteration counter
    i = 0 

   # keep going until we reach end of list (if i is above length, there is literally no list left)
    while i < len(token_ids):

        # check if there is a next token to form a pair with (if we can do token_ids[i + 1])
        if i + 1 < len(token_ids): 
            pass

        # when there is only one left you need to add that token_id below
        #####

:::{admonition} HANDS-ON
:class: red
Go through `token_ids` and add them to `new_token_ids`, only changing the `token_ids` of the token_id_pairs that matches the `pair` you are looking for. Those should have `new_id`.


In steps:
1. Pairwise check `current_token_id` and the `next_token_id` to see if they match the `pair`
2. IF they match the pair, add the `new_id`to `new_token_ids`, and add `+=2` to the iteration counter to skip that pair 
3. IF they don't match the pair, add the `current_token_id` to the `new_token_ids` and add `+1` to the iteration counter.
4. Remmeber a final "else" statement to handle when there is only one token left (with nothing to check to). It also needs to be in the `new_token_ids` 
:::

:::{admonition} DIFFICULT EXERCISE
:class: warning
If you aren't super comfortable with Python, this exercise might be prove difficult. I suggest these steps:
1. Give it 5-10 minutes of problem-solving
2. If you find it too difficult after this period, try to understand the solution
3. Go back to your notebook and see if you can reproduce the solution *without looking* at it 
4. If you get stuck, go back to the solution, repeat steps 2-3.

It can be frustrating to deal with an exercise *above* your level, but you actually learn a lot from this struggle, even if you don't suceed!
:::

#### Solution

In [11]:
def merge_pair(token_ids:list, pair:tuple, new_id:int):
    """
    Replace every copy of pair with new_id
    """
    new_token_ids = []

    # iteration counter
    i = 0 

    # keep going until we reach end of list (if i is above length, there is literally no list left)
    while i < len(token_ids):

        # get current token
        current_token = token_ids[i]

        # check if there is a next token to form a pair with (if we can do token_ids[i + 1])
        if i + 1 < len(token_ids): 
            next_token = token_ids[i + 1]
            pair_to_check = (current_token, next_token)

            if pair_to_check == pair:
                new_token_ids.append(new_id)

                # since we merged, move forward by two!
                i += 2 
            else:
                # pair doesn't match, keep current token
                new_token_ids.append(current_token)
                i += 1

        # if there is no next token, keep current token
        else: 
            new_token_ids.append(current_token)
            i += 1

    return new_token_ids

#### Test you solution.
You can copy this into the note book to do a simple test on your code works!

In [12]:
test_ids = [5, 6, 6, 7, 9, 1]
new_id = 99
pair_to_match = (6, 7)

# defining what I expect to see 
desired_result = [5, 6, 99, 9, 1]

# get result 
result = merge_pair(test_ids, (6, 7), 99)

# assert that the result matches the desired one
assert result == desired_result
print(f"Test passed. Result matches desired result.\n\nOriginal: {test_ids}.\nAfter merging {result}.\n(6, 7) -> 99:")

Test passed. Result matches desired result.

Original: [5, 6, 6, 7, 9, 1].
After merging [5, 6, 99, 9, 1].
(6, 7) -> 99:


### Step 3. Set up the Training Loop
We have a function which counts and returns the most frequent pair. We also have a function that merges said pair! We're ready for a simple training loop:

#### Step 3A: Simple Training

In [13]:
def train(token_ids, num_merges: int = 10):
    pass

:::{admonition} HANDS-ON
:class: red
Make a training loop that that: 
1. Iterates through a merge process the number of times specifed by `num_merges`
2. Gets the most frequent pair 
3. Initalizes a new id for this token id, beginning with the id 256
4. Merges based on this frequent pair
5. Save the top pair and its new id to a `merges` dictionary (look-up table) 
6. Prints the process the top pair and its new token_id
7. Returns both `token_ids` and `merges` dictionary

Again, if you find this exercise difficult after 5-15 minutes of problem-solving, there is no shame in learning from the solution! Just give it your go first :)
:::

##### Solution

In [20]:
def train(token_ids, num_merges: int = 14):
    """
    Train a BPE tokenizer on the given text.

    I modified this from:
    https://machinelearningplus.com/gen-ai/build-bpe-tokenizer/
    """
    merges = {}

    for i in range(num_merges):
        # count and find most frequent pair
        most_frequent_pair = find_most_frequent_pair(token_ids)
        
        if not most_frequent_pair:
            print("No frequent pair found, no merge")
        
        new_id = 256 + i  # new IDs start after byte range

        token_ids = merge_pair(token_ids, most_frequent_pair, new_id)
        merges[most_frequent_pair] = new_id

        print(f"Merge {i+1}: {most_frequent_pair} → {new_id} "
              f"| Tokens remaining: {len(token_ids)}")
        
    return token_ids, merges

##### Testing it
Let's try our train function:

In [23]:
input_text = "the cat sat on the mat the cat ate the rat"
token_ids = list(input_text.encode("utf-8"))
final_tokens, learned_merges = train(token_ids, num_merges=10)
print(f"\nFinal: {len(final_tokens)} tokens "
      f"(started at {len(token_ids)})")

Merge 1: (97, 116) → 256 | Tokens remaining: 36
Merge 2: (101, 32) → 257 | Tokens remaining: 31
Merge 3: (116, 104) → 258 | Tokens remaining: 27
Merge 4: (258, 257) → 259 | Tokens remaining: 23
Merge 5: (256, 32) → 260 | Tokens remaining: 19
Merge 6: (259, 99) → 261 | Tokens remaining: 17
Merge 7: (261, 260) → 262 | Tokens remaining: 15
Merge 8: (262, 115) → 263 | Tokens remaining: 14
Merge 9: (263, 260) → 264 | Tokens remaining: 13
Merge 10: (264, 111) → 265 | Tokens remaining: 12

Final: 12 tokens (started at 42)


#### Step 3B: Add "the vocab" dictionary
A real BPE would have a `vocab` dictionary with initial base bytes, adding the new id on top of this: 

In [ ]:
vocab = {i: bytes([i]) for i in range(256)} 

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

Note that some byte values are not printable characters in Python which is what you'll see with the `token_id = 0` being represented as the character `x00` (meaning Null)! The `b` in front denotes [bytes object](https://www.geeksforgeeks.org/python/effect-of-b-character-in-front-of-a-string-literal-in-python/). 

:::{admonition} HANDS-ON
:class: red
Add the `vocab` to the `train()` function. Specifically:
1. Initialize the vocab (just copy-paste the line above)
2. Make sure to add new id and the most frequent pair to the `vocab` (remember to not add it as a tuple of characters, but actually [do string concatenation](https://www.w3schools.com/python/python_strings_concatenate.asp))
3. In addition to `token_ids`, `merges`, make sure to return `vocab`
:::

#### Solution

In [24]:
def train(token_ids, num_merges: int = 14):
    """
    Train a BPE tokenizer on the given text.

    I modified this from:
    https://machinelearningplus.com/gen-ai/build-bpe-tokenizer/
    """
    merges = {}
    vocab = {i: bytes([i]) for i in range(256)}

    for i in range(num_merges):
        # count and find most frequent pair
        most_frequent_pair = find_most_frequent_pair(token_ids)
        
        if not most_frequent_pair:
            print("No frequent pair found, no merge")
        
        new_id = 256 + i  # new IDs start after byte range

        token_ids = merge_pair(token_ids, most_frequent_pair, new_id)
        merges[most_frequent_pair] = new_id

        # add to 
        vocab[new_id] = vocab[most_frequent_pair[0]] + vocab[most_frequent_pair[1]]

        print(f"Merge {i+1}: {most_frequent_pair} → {new_id} "
              f"| Tokens remaining: {len(token_ids)}")
        
    return token_ids, merges, vocab

#### Testing our solution:

In [ ]:
input_text = "the cat sat on the mat the cat ate the rat"
token_ids = list(input_text.encode("utf-8"))
final_tokens, learned_merges, vocab = train(token_ids, num_merges=10)

Merge 1: (97, 116) → 256 | Tokens remaining: 36
Merge 2: (101, 32) → 257 | Tokens remaining: 31
Merge 3: (116, 104) → 258 | Tokens remaining: 27
Merge 4: (258, 257) → 259 | Tokens remaining: 23
Merge 5: (256, 32) → 260 | Tokens remaining: 19
Merge 6: (259, 99) → 261 | Tokens remaining: 17
Merge 7: (261, 260) → 262 | Tokens remaining: 15
Merge 8: (262, 115) → 263 | Tokens remaining: 14
Merge 9: (263, 260) → 264 | Tokens remaining: 13
Merge 10: (264, 111) → 265 | Tokens remaining: 12
{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43

In [31]:
print(vocab)

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

:::{admonition} QUESTION
:class: red
Scroll through the base vocab to get to the newly added `token_ids`. 

Consider this: **How can it be so fast at its subword training** that we end up with `the cat sat` as the 9th added token?

<details>
<summary>MY ANSWER</summary>
My guess is that we're aggresively merging (merging 10 times) on a single input text. I think things would look very different if we had a larger corpus where there is a more varied (& more representative) distribution of most frequent pairs.
<br><br>
REAL BPE would also need to handle a corpus, not just a list 
</details>
:::

> The whites-spaces here are treated as their own character/byte that can be part of the subword tokenization. It seems debated how to handle that!

## 4.2 Epilogue
This exercise was inspired by the multitude of BPE tutorials out there: 
1. Sebastian Raschka's [blog](https://sebastianraschka.com/blog/2025/bpe-from-scratch.html) and [bonus material](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/05_bpe-from-scratch/bpe-from-scratch-simple.ipynb) from "Build a Large Language Model from Scratch".
2. HF's [BPE tutorial](https://huggingface.co/learn/llm-course/en/chapter6/5). 
3. MachineLearningPlus' [BPE tutorial](https://machinelearningplus.com/gen-ai/build-bpe-tokenizer/), written by Selva Prabhakaran.

> You'll notice that there are many many ways to build BPE :). I've sometimes simplified a little bit (e.g., with for loops instead of list comprehensions or not using [zip](https://www.codecademy.com/article/python-zip-function)). Feel free to explore in the tutorials!

### Expanding on this tutorial! 
Quite deep into making this exercise, I realized that this would become SUPER long given all the various steps of the BPE tokenizer. 

I've decided to end it here, focusing on a simple training flow. But I had SO much fun playin' with this and would have made this tutorial SEVERAL more parts if I had the time.

:::{admonition} ADVANCED HANDS-ON: Creating a class
:class: red
A main improvement to our simple tokenizer would move all code into a `BasicBPETokenizer` [class](https://docs.python.org/3/tutorial/classes.html). 

**If you are an advanced user of Python, familiar with classes**, I suggest trying this out: 
1. Look at the basic class structure in [Raschka's bonus material](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/05_bpe-from-scratch/bpe-from-scratch-simple.ipynb) & [MachineLearningPlus](https://machinelearningplus.com/gen-ai/build-bpe-tokenizer/)in & **adapt our train code** to this! 
    - My personal opinion is that the Raschka's `train()` method is too long & should have been split into helper functions like we've done today. Consider keeping our structure!
    - Consider adding ways to handle special characters and white-space like Rasckha does. (NB. how to handle white-space seems debated).
    
**If you aren't comfortable with classes** 
- DON'T WORRY! I'll introduce you to classes in a beginner friendly way in class 6, where we'll build a chatbot!
:::

#### "A more sophisticated" BPE
The BPE tokenizer that we built is the "simple" or "basic" edition, compared to a "real" BPE tokenizer. Rasckha actually has two tutorials:
1. ["Simple"](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/05_bpe-from-scratch/bpe-from-scratch-simple.ipynb) 
2. ["Sophisticated"](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/05_bpe-from-scratch/bpe-from-scratch.ipynb).

From an intial quick glance, I can't tell you the difference, but I would encourage you to compare them! You might also enjoy looking at [OpenAI's original GPT-2 tokenizer](https://github.com/openai/gpt-2/blob/master/src/encoder.py).